# 07 — Human-in-the-Loop

**Learning objective:** use a real graph interrupt to pause persistent execution and resume with approve, reject, or bounded revise decisions.

A human gate is useful when an action is high-impact or governed by policy. It is not `input()` inside a temporary process: the workflow state must survive while no worker is running.

## Mental model and topology

```mermaid
flowchart TD
    accTitle: Persistent human approval workflow
    accDescr: A deterministic risk policy sends high-risk proposals to an interrupt where a person may approve, reject, or request a bounded revision.

    generate[Generate proposal] --> assess[Assess structured risk]
    assess --> policy{Approval policy}
    policy -->|Low risk| execute[Execute]
    policy -->|High risk| review{{Interrupt for review}}
    review -->|Approve| execute
    review -->|Reject| stop([Stop])
    review -->|Revise within budget| revise[Revise]
    revise --> generate
    execute --> complete([End])
```

## State, nodes, and policy

State carries the proposal, structured `risk_level`, approval decision, requested changes, revision count, and trace. Generation and assessment compute values; deterministic code maps `high` risk to mandatory review. The human may choose only `approve`, `reject`, or `revise`, and revisions are bounded.

In [1]:
from langgraph.types import Command
from graph_engineering.human_loop import build_human_loop_graph, human_thread_config

graph, memory = build_human_loop_graph()
config = human_thread_config("lesson-07-approve")

In [2]:
paused = graph.invoke(
    {"request": "publish new pricing", "risk_level": "high", "trace": []},
    config,
)
print("Paused at:", graph.get_state(config).next)
print("Protected action executed:", paused.get("action_executed", False))

Paused at: ('human_review',)
Protected action executed: False


In [3]:
approved = graph.invoke(Command(resume={"decision": "approve"}), config)
print("Decision:", approved["decision"])
print("Action executed:", approved["action_executed"])
print("Termination:", approved["termination_reason"])

Decision: approve
Action executed: True
Termination: approved_and_executed


## Rejection, revision, and failure policy

Rejection terminates without the protected action. Revision changes the proposal and returns to review, but only within `MAX_REVISIONS`. Unknown decisions follow the safe stop path. In production, also define reviewer authentication, timeout, reassignment, stale-approval checks, and an idempotency key for execution.

In [4]:
reject_config = human_thread_config("lesson-07-reject")
graph.invoke({"request": "delete records", "risk_level": "high", "trace": []}, reject_config)
rejected = graph.invoke(Command(resume={"decision": "reject"}), reject_config)
print("Rejected action executed:", rejected["action_executed"])
print("Termination:", rejected["termination_reason"])

Rejected action executed: False
Termination: human_rejected


In [5]:
revise_config = human_thread_config("lesson-07-revise")
graph.invoke({"request": "publish report", "risk_level": "high", "trace": []}, revise_config)
repaused = graph.invoke(Command(resume={"decision": "revise", "requested_changes": "remove personal data"}), revise_config)
print("Revisions:", repaused["revisions"])
print("Revised proposal:", repaused["proposal"])
print("Paused again at:", graph.get_state(revise_config).next)

Revisions: 1
Revised proposal: Proposal for publish report; revised with: remove personal data
Paused again at: ('human_review',)


## What to modify

Change `risk_level` to `low` and observe that deterministic policy bypasses review. Then adjust the revision bound or review packet—not the hard policy via an LLM prompt.

**Next:** [08 — Observability and Tracing](08_observability_and_tracing.ipynb) makes these control decisions measurable.